In [ ]:
import cv2
import numpy as np

# Load OpenPose model files
protoFile = "models/pose_deploy.prototxt"
weightsFile = "models/pose_iter_440000.caffemodel"

# Load the network
net = cv2.dnn.readNetFromCaffe(protoFile, weightsFile)

# Load an image
image = cv2.imread("image.png")
height, width, _ = image.shape

# Prepare the input image
inpBlob = cv2.dnn.blobFromImage(image, 1.0 / 255, (368, 368), (0, 0, 0), swapRB=False, crop=False)
net.setInput(inpBlob)

# Run forward pass to get output
output = net.forward()

# Define body keypoints for COCO model
BODY_PARTS = {
    0: "Nose", 1: "Neck", 2: "RShoulder", 3: "RElbow", 4: "RWrist",
    5: "LShoulder", 6: "LElbow", 7: "LWrist", 8: "RHip", 9: "RKnee",
    10: "RAnkle", 11: "LHip", 12: "LKnee", 13: "LAnkle", 14: "REye",
    15: "LEye", 16: "REar", 17: "LEar"
}

# Define threshold to filter weak keypoints
threshold = 0.1

# Draw detected keypoints and print their confidence and heatmap coordinates
for i in range(len(BODY_PARTS)):  
    heatMap = output[0, i, :, :]
    _, conf, _, point = cv2.minMaxLoc(heatMap)

    x = int((width * point[0]) / output.shape[3])
    y = int((height * point[1]) / output.shape[2])

    print(f"Body Part: {BODY_PARTS[i]}, Confidence: {conf:.4f}, Heatmap Coordinates: {point}, Mapped Coordinates: ({x}, {y})")

    if conf > threshold:
        cv2.circle(image, (x, y), 5, (0, 255, 0), thickness=-1, lineType=cv2.FILLED)
        cv2.putText(image, BODY_PARTS[i], (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

# Show the output image
cv2.imshow("Pose Estimation", image)
cv2.waitKey(0)
cv2.destroyAllWindows()


error: OpenCV(4.11.0) D:\a\opencv-python\opencv-python\opencv\modules\dnn\src\caffe\caffe_io.cpp:1126: error: (-2:Unspecified error) FAILED: fs.is_open(). Can't open "models/pose_deploy.prototxt" in function 'cv::dnn::ReadProtoFromTextFile'


In [2]:
import cv2
import mediapipe as mp
import numpy as np

# Initialize MediaPipe Pose
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
pose = mp_pose.Pose()

# List of image paths (Replace with your actual image file names)
image_paths = ["hand1.jpg"]

for img_path in image_paths:
    # Read the image
    image = cv2.imread(img_path)
    if image is None:
        print(f"Error loading {img_path}")
        continue

    # Convert BGR to RGB (required for MediaPipe)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Process the image
    results = pose.process(image_rgb)

    # Draw landmarks if detected
    if results.pose_landmarks:
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        
        # Print landmark coordinates
        print(f"Keypoints for {img_path}:")
        for idx, landmark in enumerate(results.pose_landmarks.landmark):
            print(f"Landmark {idx}: (X: {landmark.x}, Y: {landmark.y}, Z: {landmark.z})")

    # Display the image with landmarks
    cv2.imshow(f'Pose Estimation - {img_path}', image)
    cv2.waitKey(0)  # Wait for key press before closing the image

cv2.destroyAllWindows()

Error loading hand1.jpg
